# Nudity Unlearning Multi-Benchmark Demo

This notebook runs a **MultiBenchmarkRunner** for each nudity-compatible unlearning technique against the full suite of nudity evaluation metrics: `asr`, `err`, `fid`, `clip_score`, and `ua_ira`.

Each technique is run in its own cell. Configs are loaded from `examples/demo configs/`.

In [ ]:
import gc
import os
import json
import torch
from dotenv import load_dotenv

load_dotenv(override=True)

from eval_learn.runners import MultiBenchmarkRunner
from eval_learn.logging_utils import get_logger

logger = get_logger(__name__)


def load_config(path: str) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def build_multi(config: dict) -> MultiBenchmarkRunner:
    tech = config["technique"]
    metric_names = [m["name"] for m in config["metrics"]]
    metric_configs = {m["name"]: m["config"] for m in config["metrics"] if m.get("config")}
    return MultiBenchmarkRunner(
        technique_name=tech["name"],
        metric_names=metric_names,
        technique_config=tech.get("config", {}),
        metric_configs=metric_configs,
        output_dir=config.get("output_dir", "results"),
    )


def print_results(report: dict):
    print(f"Run ID: {report.get('run_id', 'N/A')}")
    for name, r in report.get("metric_results", {}).items():
        print(f"  {r['name']}: {r['value']}")


def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

c:\ProgramData\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. ESD — Erased Stable Diffusion

Trains the UNet cross-attention layers to erase the nudity concept. Uses `train_method=noxattn` (fine-tunes all layers except cross-attention), which is recommended for general concepts like nudity.

In [ ]:
config = load_config("examples/demo configs/esd_nudity_multi.json")
runner = build_multi(config)
report = runner.run()
print_results(report)
del runner
cleanup()

2026-03-27 11:25:58 - eval_learn.registry.entrypoints - INFO - Loading plugins from entry points...


C:\Users\nikhi\AppData\Roaming\Python\Python313\site-packages\diffusers\models\transformers\transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
C:\Users\nikhi\AppData\Roaming\Python\Python313\site-packages\diffusers\models\transformers\transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)


2026-03-27 11:26:15 - eval_learn.registry.entrypoints - ERROR - Failed to load plugin concept_steerers: ConceptSteerersWrapper requires the 'concept-steerers' package. Package not installed.
2026-03-27 11:26:15 - eval_learn.registry.entrypoints - ERROR - Failed to load plugin esd: ConceptSteerersWrapper requires the 'concept-steerers' package. Package not installed.
2026-03-27 11:26:15 - eval_learn.registry.entrypoints - ERROR - Failed to load plugin mace: ConceptSteerersWrapper requires the 'concept-steerers' package. Package not installed.
2026-03-27 11:26:15 - eval_learn.registry.entrypoints - ERROR - Failed to load plugin safree: ConceptSteerersWrapper requires the 'concept-steerers' package. Package not installed.
2026-03-27 11:26:15 - eval_learn.registry.entrypoints - INFO - Registered plugin 'sld' (eval_learn.techniques)
2026-03-27 11:26:15 - eval_learn.registry.entrypoints - INFO - Registered plugin 'uce' (eval_learn.techniques)
2026-03-27 11:26:17 - eval_learn.registry.entrypo

ValueError: Technique 'esd' not found. Available: ['sld', 'uce']

## 2. MACE — Mass Concept Erasure

Uses Closed-Form Refinement (CFR) to analytically update K/V projection matrices in every cross-attention layer — no training loop required. `lambda_cfr` controls the regularisation strength.

In [ ]:
config = load_config("examples/demo configs/mace_nudity_multi.json")
runner = build_multi(config)
report = runner.run()
print_results(report)
del runner
cleanup()

## 3. UCE — Unlearning with Concept Erasure

Loads a pre-trained nudity-erased checkpoint via `preset=nudity`. Note: UCE uses the `preset` field instead of `erase_concept` directly; the concept is derived from the preset at validation time.

In [ ]:
config = load_config("examples/demo configs/uce_nudity_multi.json")
runner = build_multi(config)
report = runner.run()
print_results(report)
del runner
cleanup()

## 4. SAeUron — Sparse Autoencoder Unlearning

Hooks into a specific UNet layer and ablates SAE features associated with nudity. `sae_path` and `acts_path` are auto-resolved from the installed package if not specified.

In [ ]:
config = load_config("examples/demo configs/saeuron_nudity_multi.json")
runner = build_multi(config)
report = runner.run()
print_results(report)
del runner
cleanup()

## 5. SLD — Safe Latent Diffusion

Uses the `AIML-TUDA/stable-diffusion-safe` model with a safety guidance mechanism applied at inference time. `preset=max` applies the strongest suppression parameters.

In [ ]:
config = load_config("examples/demo configs/sld_nudity_multi.json")
runner = build_multi(config)
report = runner.run()
print_results(report)
del runner
cleanup()

## 6. SAFREE — Selective and Attribute Free

A training-free three-stage pipeline: Text Projection, Self-Validation Filter (SVF), and Latent Re-Attention (LRA). All stages are tuned for nudity erasure.

In [ ]:
config = load_config("examples/demo configs/safree_nudity_multi.json")
runner = build_multi(config)
report = runner.run()
print_results(report)
del runner
cleanup()

## 7. ConceptSteerers

Steers generation away from nudity using a Sparse Autoencoder checkpoint trained on the I2P dataset. `sae_path` is auto-resolved from the installed package if not specified.

In [ ]:
config = load_config("examples/demo configs/concept_steerers_nudity_multi.json")
runner = build_multi(config)
report = runner.run()
print_results(report)
del runner
cleanup()